<p align="center"><a href="https://www.plus2net.com/python/pandas-to_clipboard.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Pandas to_clipboard(): Copy DataFrames and Preview Spreadsheet Text

Run cells in order. Data is embedded and synthetic. Portable text/file examples run in Colab; native clipboard calls are desktop-only and disabled by default. Colab cannot directly access your Windows clipboard. Excel examples need openpyxl. Temporary files are cleaned automatically. Try the exercise before its solution.

## Preview the original DataFrame as tab-separated text

These synthetic original records retain ENGLISH marks 20, 70 and 41. to_clipboard() writes text to the system clipboard and returns None. In Colab, preview the same tab-separated representation with to_csv(), then copy the displayed text yourself or download it.

In [ ]:
import pandas as pd
import tempfile
import sqlite3
from pathlib import Path
from io import StringIO
df = pd.DataFrame({"NAME": ["Ravi", "Raju", "Alex"], "ID": [1, 2, 3],
    "MATH": [30, 40, 50], "ENGLISH": [20, 70, 41]})
text = df.to_csv(sep="	")
print(text.rstrip())
restored = pd.read_csv(StringIO(text), sep="	", index_col=0)
pd.testing.assert_frame_equal(restored, df)

**Expected output**

```text
	NAME	ID	MATH	ENGLISH
0	Ravi	1	30	20
1	Raju	2	40	70
2	Alex	3	50	41
```

## Copy a DataFrame in local Python

Run locally, set run_desktop=True, then paste into spreadsheet cells. excel=True formats delimited text suitable for spreadsheet paste; it does not create an .xlsx workbook. This optional cell is disabled in portable Run All and will replace your clipboard when enabled.

In [ ]:
run_desktop = False  # Enable only when you want to replace the local clipboard.
if run_desktop:
    result = df.to_clipboard(index=False)
    assert result is None
    print("Copied rows:", len(df))
else:
    print("Desktop clipboard write is disabled; use text or file exports.")

**Expected output**

```text
Desktop clipboard write is disabled; use text or file exports.
```

## Omit the index when pasting

The default delimited output includes the index. For ordinary spreadsheet rows, pass index=False. This portable preview matches <code>df.to_clipboard(index=False)</code>.

In [ ]:
text = df.to_csv(sep="	", index=False)
print(text.rstrip())
pd.testing.assert_frame_equal(pd.read_csv(StringIO(text), sep="	"), df)

**Expected output**

```text
NAME	ID	MATH	ENGLISH
Ravi	1	30	20
Raju	2	40	70
Alex	3	50	41
```

## Choose comma-separated output

Retain the original sep="," example. On desktop, use <code>df.to_clipboard(index=False, sep=",")</code>. Commas may need a spreadsheet import option instead of direct cell paste; tabs are the usual spreadsheet delimiter.

In [ ]:
text = df.to_csv(index=False, sep=",")
print(text.rstrip())
pd.testing.assert_frame_equal(pd.read_csv(StringIO(text)), df)

**Expected output**

```text
NAME,ID,MATH,ENGLISH
Ravi,1,30,20
Raju,2,40,70
Alex,3,50,41
```

## Use excel=False for display text

excel=False copies a DataFrame string representation instead of CSV-like text. It is intended for readable display rather than predictable spreadsheet cells. The preview uses to_string(index=False); the equivalent local call is <code>df.to_clipboard(excel=False, index=False)</code>.

In [ ]:
display_text = df.to_string(index=False)
print(display_text)
assert "Ravi" in display_text and "ENGLISH" in display_text

**Expected output**

```text
NAME  ID  MATH  ENGLISH
Ravi   1    30       20
Raju   2    40       70
Alex   3    50       41
```

## Copy selected rows and columns

Select the table before exporting. Or use the columns keyword with excel=True. This preview matches a desktop copy of NAME and MATH for marks at least 40.

In [ ]:
selected = df.loc[df["MATH"].ge(40), ["NAME", "MATH"]].copy()
text = selected.to_csv(sep="	", index=False)
print(text.rstrip())
assert pd.read_csv(StringIO(text), sep="	")["NAME"].tolist() == ["Raju", "Alex"]

**Expected output**

```text
NAME	MATH
Raju	40
Alex	50
```

## Control missing values and decimal display

With excel=True, CSV formatting keywords such as na_rep and float_format are forwarded to to_csv(). In local Python the same arguments work with to_clipboard(). These synthetic amounts use NA text deliberately; agree on its meaning with the recipient.

In [ ]:
amounts = pd.DataFrame({"id": ["001", "002"], "amount": [5.25, None]})
text = amounts.to_csv(sep="	", index=False, na_rep="NA", float_format="%.2f")
print(text.rstrip())
checked = pd.read_csv(StringIO(text), sep="	", dtype={"id": "string"})
assert checked["id"].tolist() == ["001", "002"]
assert checked["amount"].isna().sum() == 1

**Expected output**

```text
id	amount
001	5.25
002	NA
```

## Export records from a database

Preserve the original MySQL/database-to-clipboard pipeline with a self-contained SQLite version. Query first, then format the DataFrame for copying. For live MySQL use a compatible SQLAlchemy engine and driver with a managed connection and private credentials; raw non-SQLite DBAPI connections are not generally supported by Pandas. See <a href="mysql-connection.php">MySQL connections</a>, <a href="mysql-sqlalchemy.php">SQLAlchemy</a>, <a href="pandas-mysql.php">MySQL read_sql()</a> and <a href="pandas-read_sql.php">SQLite read_sql()</a>.

In [ ]:
connection = sqlite3.connect(":memory:")
try:
    df.to_sql("student", connection, index=False, if_exists="replace")
    rows = pd.read_sql("SELECT * FROM student ORDER BY ID LIMIT 5", connection)
finally:
    connection.close()
text = rows.to_csv(sep="	", index=False)
print(text.rstrip())
pd.testing.assert_frame_equal(rows, df)
# Local Python, when clipboard access is available:
# rows.to_clipboard(index=False)

**Expected output**

```text
NAME	ID	MATH	ENGLISH
Ravi	1	30	20
Raju	2	40	70
Alex	3	50	41
```

## Export Excel and CSV sources

Retain both source workflows. This creates small source files rather than relying on machine-specific paths. After importing, local Python can use <code>imported.to_clipboard(index=False)</code>. In Colab, print or download tab-separated text. See <a href="pandas-read_excel.php">read_excel()</a> and <a href="pandas-read_csv.php">read_csv()</a>.

In [ ]:
with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
    csv_file = Path(folder) / "student.csv"
    excel_file = Path(folder) / "student.xlsx"
    df.to_csv(csv_file, index=False)
    df.to_excel(excel_file, index=False, engine="openpyxl")
    for imported in (pd.read_csv(csv_file), pd.read_excel(excel_file, engine="openpyxl")):
        pd.testing.assert_frame_equal(imported, df)
        assert imported.to_csv(sep="	", index=False) == df.to_csv(sep="	", index=False)
    print("CSV and Excel source previews passed")

**Expected output**

```text
CSV and Excel source previews passed
```

## Questions and clipboard choices

<ol><li><strong>Purpose?</strong> Copy a DataFrame text representation to the operating-system clipboard.</li><li><strong>Basic use?</strong> Run df.to_clipboard() in local Python with clipboard access.</li><li><strong>Parameters?</strong> excel, sep and formatting keyword arguments.</li><li><strong>Example?</strong> df.to_clipboard(index=False) omits row labels.</li><li><strong>Format?</strong> Tab-separated delimited text by default, including the index.</li><li><strong>Subset?</strong> Select with loc before copying, or supply columns for delimited output.</li><li><strong>Missing values?</strong> na_rep controls their text representation.</li><li><strong>Alternatives?</strong> to_csv(), to_excel() and manual copy of displayed text.</li><li><strong>OS interaction?</strong> The clipboard belongs to the environment where Python runs, not a remote browser client.</li><li><strong>Customize?</strong> Choose separator, index, header, columns and CSV formatting; excel=False produces display text.</li></ol><p>Colab cannot directly write to your Windows clipboard. A missing desktop backend can raise PyperclipException. Avoid copying sensitive tables accidentally and remember that clipboard contents can be replaced. For large transfers use a file. Exporting spreadsheet-ready text does not validate how every spreadsheet will interpret dates, zeros or formula-like strings; set destination types appropriately.</p>

## Exercise: export a selected table

Create tab-separated NAME and ENGLISH text for records with ENGLISH at least 40, without an index. Verify Raju and Alex. Optional desktop copy is a separate action.

## Exercise solution

Filter and preview the exact text before using a clipboard or file export.

In [ ]:
answer = df.loc[df["ENGLISH"].ge(40), ["NAME", "ENGLISH"]].copy()
text = answer.to_csv(sep="	", index=False)
print(text.rstrip())
assert pd.read_csv(StringIO(text), sep="	")["NAME"].tolist() == ["Raju", "Alex"]
# Local Python only: answer.to_clipboard(index=False)
# Colab file alternative:
# answer.to_csv("selected.tsv", sep="	", index=False)
# from google.colab import files
# files.download("selected.tsv")

**Expected output**

```text
NAME	ENGLISH
Raju	70
Alex	41
```